**Generating test datasets** | https://academy.claude.com/courses/building-with-the-claude-api/generating-test-datasets

In [1]:
import boto3
import json
import re

client = boto3.client("bedrock-runtime", region_name="us-east-1")

model_id = "us.anthropic.claude-sonnet-4-6"

def add_user_message(messages, text):
    user_message = {
        "role": "user",
        "content": [
            {"text": text}
        ],
    }
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {
        "role": "assistant",
        "content": [
            {"text": text}
        ],
    }
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, max_tokens=2000, stop_sequences=None, DEBUG=False):

    params = {
        "modelId": model_id,
        "messages": messages,
        "inferenceConfig": {
            "temperature": temperature,
            "maxTokens": max_tokens,
        }
    }

    if stop_sequences:
        params["inferenceConfig"]["stopSequences"] = stop_sequences

    if system:
        params["system"] = [{"text": system}]

    response = client.converse(**params)


    if DEBUG:
        print("=========[ RAW RESPONSE ]=========")
        print(json.dumps(response, indent=2))
        print("========= >> ========= >> ========= >> ========= >>")

    content = response["output"]["message"]["content"]
    if content:
        return content[0]["text"]
    return ""


In [2]:
def generate_dataset():
    prompt = """
Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

Example output:
```json
[
    {
        "task": "Description of task",
    },
    ...additional
]
```

* Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
* Focus on tasks that do not require writing much code

Please generate 3 objects.
"""

    messages = []
    add_user_message(messages, prompt)
    text = chat(messages)
    # return json.loads(text)
    return extract_json(text)


def extract_json(text):
    """Pull JSON out of the reply, even if the model wrapped it in ```json fences."""
    match = re.search(r"```(?:json)?\s*(.*?)```", text, re.DOTALL)
    if match:
        text = match.group(1)
    return json.loads(text.strip())

In [3]:
dataset = generate_dataset()
print(json.dumps(dataset, indent=2))

if dataset:
    with open("dataset.json", "w") as f:
        json.dump(dataset, f, indent=2)
    print(f"Saved {len(dataset)} tasks to dataset.json")
else:
    print("The list is completely empty.")



[
  {
    "task": "Write a Python function that takes an AWS S3 bucket name and a prefix string as arguments and returns a list of all object keys in that bucket matching the prefix, handling pagination automatically using boto3."
  },
  {
    "task": "Write a JSON object representing an AWS IAM policy that allows a user to only list and get objects from a specific S3 bucket named 'my-app-assets', but explicitly denies the ability to delete any objects."
  },
  {
    "task": "Write a regex pattern that validates an AWS ARN (Amazon Resource Name), ensuring it matches the standard format such as 'arn:aws:iam::123456789012:user/JohnDoe' or 'arn:aws:s3:::my-bucket'."
  }
]
Saved 3 tasks to dataset.json


In [4]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""
    prompt = f"""
Please solve the following task:

{test_case["task"]}
"""
    messages = []
    add_user_message(messages, prompt)
    output = chat(messages, DEBUG=True)
    return output

def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""
    output = run_prompt(test_case)

    # TODO - Grading
    score = 10

    return {
        "output": output,
        "test_case": test_case,
        "score": score
    }

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""
    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)

    return results

In [5]:
with open("dataset.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

=========[ RAW RESPONSE ]=========
{
  "ResponseMetadata": {
    "RequestId": "904ccf2d-3632-42f5-8903-8664603d7230",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "date": "Thu, 15 Oct 2026 21:26:22 GMT",
      "content-type": "application/json",
      "content-length": "7115",
      "connection": "keep-alive",
      "x-amzn-requestid": "904ccf2d-3632-42f5-8903-8664603d7230"
    },
    "RetryAttempts": 0
  },
  "output": {
    "message": {
      "role": "assistant",
      "content": [
        {
          "text": "# AWS S3 Object Keys Retrieval with Pagination\n\n## Solution\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\nfrom typing import Optional\n\n\ndef list_s3_objects(\n    bucket_name: str,\n    prefix: str = \"\",\n    region_name: Optional[str] = None,\n    max_keys_per_page: int = 1000,\n) -> list[str]:\n    \"\"\"\n    Retrieve all object keys from an S3 bucket matching a given prefix.\n\n    Handles pagination automaticall

In [6]:
print(json.dumps(results, indent=2))

[
  {
    "output": "# AWS S3 Object Keys Retrieval with Pagination\n\n## Solution\n\n```python\nimport boto3\nfrom botocore.exceptions import ClientError, NoCredentialsError\nfrom typing import Optional\n\n\ndef list_s3_objects(\n    bucket_name: str,\n    prefix: str = \"\",\n    region_name: Optional[str] = None,\n    max_keys_per_page: int = 1000,\n) -> list[str]:\n    \"\"\"\n    Retrieve all object keys from an S3 bucket matching a given prefix.\n\n    Handles pagination automatically to ensure all objects are returned,\n    regardless of how many exist in the bucket.\n\n    Args:\n        bucket_name:       The name of the S3 bucket to query.\n        prefix:            The prefix string to filter object keys.\n                           Defaults to \"\" (matches all objects).\n        region_name:       AWS region name (e.g., \"us-east-1\").\n                           If None, uses the default configured region.\n        max_keys_per_page: Maximum number of keys to retrieve pe